In [2]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm

class RadicalDataset(Dataset):
    def __init__(self, folder):
        self.paths = [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".png")]
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=3),
            transforms.RandomResizedCrop(224, scale=(0.4, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.4, 0.4, 0.2, 0.1),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("L")
        return self.transform(img), self.transform(img)

class DINOLoss(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        self.temp_student = 0.1
        self.temp_teacher = 0.07
        self.center_momentum = 0.9
        self.register_buffer("center", torch.zeros(1, out_dim))

    def forward(self, student_out, teacher_out):
        student_out = torch.log_softmax(student_out / self.temp_student, dim=-1)
        teacher_out = torch.softmax((teacher_out - self.center) / self.temp_teacher, dim=-1)
        loss = -torch.sum(teacher_out * student_out, dim=-1).mean()
        self.center = self.center * self.center_momentum + teacher_out.mean(dim=0, keepdim=True) * (1 - self.center_momentum)
        return loss

class DINOModel(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(pretrained=True)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])  # Remove FC layer
        self.projector = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 2048),
            nn.GELU(),
            nn.Linear(2048, 2048),
            nn.GELU(),
            nn.Linear(2048, 512),
        )

    def forward(self, x):
        x = self.backbone(x)  # shape: (batch, 512, 1, 1)
        x = self.projector(x)
        return x

if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running on {device}")

    dataset = RadicalDataset("/app/data/radicals")
    loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=0)

    student = DINOModel().to(device)
    teacher = DINOModel().to(device)
    teacher.load_state_dict(student.state_dict())
    for p in teacher.parameters():
        p.requires_grad = False

    criterion = DINOLoss(out_dim=512)
    optimizer = torch.optim.Adam(student.parameters(), lr=1e-4)

    for epoch in range(10):
        student.train()
        total_loss = 0
        for view1, view2 in tqdm(loader, desc=f"Epoch {epoch+1}/10"):
            view1, view2 = view1.to(device), view2.to(device)
            student_out = student(view1)
            with torch.no_grad():
                teacher_out = teacher(view2)
            loss = criterion(student_out, teacher_out)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

    torch.save(student.state_dict(), "/app/notebooks/dino_radicals_resnet.pth")
    print("✅ Model saved to /app/notebooks/dino_radicals_resnet.pth")


Running on cpu
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


/usr/local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 44.7M/44.7M [00:01<00:00, 44.8MB/s]
Epoch 1/10: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 62/62 [00:29<00:00,  2.08it/s]


Epoch 1: Loss = 369.9568


Epoch 2/10: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 62/62 [00:28<00:00,  2.20it/s]


Epoch 2: Loss = 368.7527


...

Epoch 8: Loss = 368.2694


Epoch 9/10: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 62/62 [00:31<00:00,  1.97it/s]


Epoch 9: Loss = 368.0562


Epoch 10/10: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 62/62 [00:33<00:00,  1.88it/s]

Epoch 10: Loss = 367.9059
✅ Model saved to /app/notebooks/dino_radicals_resnet.pth


In [ ]:
# File: notebooks/extract_kanji_features.py

import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import numpy as np

class KanjiDataset(Dataset):
    def __init__(self, folder):
        self.paths = [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".png")]
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=3),
            transforms.Resize(224),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = self.paths[idx]
        img = Image.open(img_path).convert("L")
        img = self.transform(img)
        return img, os.path.basename(img_path)

class DINOFeatureExtractor(torch.nn.Module):
    def __init__(self, model_path):
        super().__init__()
        from torchvision import models
        resnet = models.resnet18(pretrained=False)
        self.backbone = torch.nn.Sequential(*list(resnet.children())[:-1])
        self.projector = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(512, 2048),
            torch.nn.GELU(),
            torch.nn.Linear(2048, 2048),
            torch.nn.GELU(),
            torch.nn.Linear(2048, 512),
        )
        self.load_state_dict(torch.load(model_path))
        self.eval()

    def forward(self, x):
        x = self.backbone(x)
        x = self.projector(x)
        return x

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_path = "/app/notebooks/dino_radicals_resnet.pth"
    kanji_folder = "/app/data/jouyou"
    output_folder = "/app/notebooks/kanji_features"
    os.makedirs(output_folder, exist_ok=True)

    dataset = KanjiDataset(kanji_folder)
    loader = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=0)

    model = DINOFeatureExtractor(model_path).to(device)

    with torch.no_grad():
        for imgs, names in tqdm(loader, desc="Extracting Kanji Features"):
            imgs = imgs.to(device)
            feats = model(imgs).cpu().numpy()
            for f, name in zip(feats, names):
                np.save(os.path.join(output_folder, name.replace(".png", ".npy")), f)

    print(f"✅ Features saved in {output_folder}")


/usr/local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Extracting Kanji Features: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 267/267 [00:38<00:00,  6.94it/s]

✅ Features saved in /app/notebooks/kanji_features


: 